In [3]:
import pandas as pd
import os

filepath = "../DATA-HTML-STOCK/NEPSECompany/NEPSECompanyExtractor.csv"
symbols = pd.read_csv(filepath)['Symbol'].tolist()

stock_ignored = ["CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "AMFIPO", "NMLBS", "ILFCPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]


def detailed_prediction(score):
    if score >= 0.60: return "Strongly Positive"
    elif score >= 0.20: return "Positive"
    elif score >= 0.05: return "Slightly Positive"
    elif score > -0.05: return "Neutral"
    elif score > -0.20: return "Slightly Negative"
    elif score > -0.60: return "Negative"
    else: return "Strongly Negative"


def apply_decay_with_memory(df, last_scores=None):
    df = df.sort_values("Date_Stock").reset_index(drop=True)

    decayed = []

    
    if last_scores:
        t1, t2, t3 = last_scores
    else:
        t1, t2, t3 = 0.0, 0.0, 0.0

    for _, row in df.iterrows():
        score = row["Sentiment_Score"]

        t3 = t2 / 2
        t2 = t1 / 2
        t1 = score

        decayed.append(t1 + t2 + t3)

    df["Decayed_Sentiment"] = decayed
    df["Decayed_Prediction"] = df["Decayed_Sentiment"].apply(detailed_prediction)

    return df, (t1, t2, t3)


for symbol in symbols:

    # if symbol not in COMPACTSTOCK:
    #     continue
    
    if symbol in stock_ignored:
        continue

    semifinal_path = f"../DATA-HTML-STOCK/SemiFinalDataset/{symbol}.csv"
    savepath = f"../DATA-HTML-STOCK/FinalDataSet/{symbol}.csv"

    if not os.path.exists(semifinal_path):
        continue

    semifinal_df = pd.read_csv(semifinal_path)
    semifinal_df["Date_Stock"] = pd.to_datetime(semifinal_df["Date_Stock"])

    if os.path.exists(savepath):

        old_df = pd.read_csv(savepath)
        old_df["Date_Stock"] = pd.to_datetime(old_df["Date_Stock"])

        saved_date = old_df["Date_Stock"].max()

        new_rows = semifinal_df[semifinal_df["Date_Stock"] > saved_date]

        if new_rows.empty:
            print(f"{symbol} no new data")
            continue

        
        old_sorted = old_df.sort_values("Date_Stock")

        last_scores = (
            old_sorted["Sentiment_Score"].iloc[-1],
            old_sorted["Sentiment_Score"].iloc[-2] / 2 if len(old_sorted) > 1 else 0.0,
            old_sorted["Sentiment_Score"].iloc[-3] / 4 if len(old_sorted) > 2 else 0.0
        )

        new_rows, _ = apply_decay_with_memory(new_rows, last_scores)

        final_df = pd.concat([old_df, new_rows], ignore_index=True)

    else:
        if symbol in stock_no_news:
            semifinal_df["Decayed_Sentiment"] = 0.0
            semifinal_df["Decayed_Prediction"] = "Neutral"
            final_df = semifinal_df
        else:
            final_df, _ = apply_decay_with_memory(semifinal_df)

    
    final_df = final_df.drop_duplicates(subset=["Date_Stock"])
    final_df = final_df.sort_values("Date_Stock", ascending=False).reset_index(drop=True)

    final_df.to_csv(savepath, index=False)

    print(f"{symbol} done: {len(final_df)} rows")
print(f"All done")

ADBL done: 3523 rows
API done: 2356 rows
HATH done: 716 rows
HATHPO done: 1 rows
AKPL done: 2004 rows
AHPC done: 3616 rows
ALICL done: 3482 rows
ALICLP done: 73 rows
BOKL done: 1321 rows
BOKLPO done: 42 rows
BARUN done: 2281 rows
BFC done: 1788 rows
BFCPO done: 28 rows
BHBL done: 877 rows
BHBLPO done: 9 rows
BBC done: 2236 rows
BNT done: 1908 rows
BNL done: 392 rows
BUDBL done: 600 rows
BUDBLP done: 4 rows
BPCL done: 3621 rows
CFCL done: 2802 rows
CFCLPO done: 4 rows
CCBL done: 1892 rows
CCBLPO done: 143 rows
CBBL done: 3113 rows
CHL done: 1904 rows
CHCL done: 4309 rows
CIT done: 2950 rows
CZBIL done: 3777 rows
CZBILP done: 122 rows
CBL done: 1894 rows
CBLPO done: 106 rows
HLBSL done: 2284 rows
CORBL done: 1823 rows
DDBL done: 3094 rows
DDBLPO done: 4 rows
DBBL done: 1009 rows
DBBLPO done: 17 rows
DHPL done: 1969 rows
EBL done: 3406 rows
EBLPO done: 2 rows
EBLCP done: 804 rows
EIC done: 1726 rows
EDBL done: 2997 rows
EDBLPO done: 42 rows
FMDBL done: 2781 rows
FMDBLP done: 30 rows
FOWAD